# Clasificación de números de base de datos MNIST con Torch [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1bjJujJNXHSQzrBExLtiwOIKx9artwm2F#scrollTo=cell-00)

**Transparencia:** El preprocesamiento de los datos se generó con Claude.

Preprocesamiento de datos: secciones 1.-3. Sección 4. contiene el ejercicio en sí.

**Instrucciones:**
1. File > Save copy in Drive (para tener su propia copia)
2. En su copia, completar el código necesario en la sección 4 para entrenar al modelo.




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

## Configuration

In [ ]:
DATA_DIR = "./data"     # MNIST is downloaded here (~55 MB, only the first time)
N_VAL = 10_000          # images taken out of the 60 000 training images for validation
BATCH_SIZE = 128
SEED = 1234             # fixes the train/val split and the weight initialisation

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():   # Apple silicon
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

## 1. Download and read in the data

`torchvision` downloads the four original MNIST files and decodes them.
We only use it for that: `.data` and `.targets` give us the raw tensors,
which we then handle ourselves.

In [ ]:
mnist_train = datasets.MNIST(root=DATA_DIR, train=True, download=True)
mnist_test = datasets.MNIST(root=DATA_DIR, train=False, download=True)

X_full, y_full = mnist_train.data, mnist_train.targets   # (60000, 28, 28) uint8, (60000,) int64
X_test, y_test = mnist_test.data, mnist_test.targets     # (10000, 28, 28) uint8, (10000,) int64

print("Training images:", tuple(X_full.shape), X_full.dtype)
print("Training labels:", tuple(y_full.shape), y_full.dtype)
print("Test images:    ", tuple(X_test.shape), X_test.dtype)
print("Pixel values from", X_full.min().item(), "to", X_full.max().item())
print("Images per digit (training):", torch.bincount(y_full).tolist())

In [ ]:
# Look at a few examples
fig, axes = plt.subplots(3, 8, figsize=(10, 4))
for ax, img, label in zip(axes.flat, X_full, y_full):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(int(label))
    ax.axis("off")
fig.suptitle("Examples from the MNIST training set")
plt.tight_layout()
plt.show()

## 2. Split into training and validation sets

A random permutation of the indices, with a fixed seed so the split is the
same every time the notebook is run.

- **training set**: used to fit the weights
- **validation set**: used to compare models / choose hyperparameters
- **test set**: used once, at the end, to report the final accuracy

In [ ]:
perm = torch.randperm(len(X_full), generator=torch.Generator().manual_seed(SEED))
val_idx, train_idx = perm[:N_VAL], perm[N_VAL:]

X_train, y_train = X_full[train_idx], y_full[train_idx]
X_val, y_val = X_full[val_idx], y_full[val_idx]

print(f"train: {len(X_train)}   validation: {len(X_val)}   test: {len(X_test)}")

## 3. Preprocessing

- scale pixels from 0…255 to 0…1,
- standardise to zero mean and unit variance, **using statistics of the
  training set only** (so no information leaks from validation/test data),
- flatten each 28×28 image into a vector of length 784, which is the input a
  feedforward (fully connected) network expects.

In [ ]:
mean = (X_train.float() / 255).mean()
std = (X_train.float() / 255).std()
print(f"Training-set pixel mean = {mean:.4f}, std = {std:.4f}")

def preprocess(x):
    """uint8 images (N, 28, 28) -> standardised float vectors (N, 784)."""
    x = x.float() / 255
    x = (x - mean) / std
    return x.reshape(len(x), -1)

train_ds = TensorDataset(preprocess(X_train), y_train)
val_ds = TensorDataset(preprocess(X_val), y_val)
test_ds = TensorDataset(preprocess(X_test), y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1000)
test_loader = DataLoader(test_ds, batch_size=1000)

xb, yb = next(iter(train_loader))
print("One batch:", tuple(xb.shape), tuple(yb.shape))

## 4. Ahora sí, a entrenar

Vean la documentación de
[`nn.Sequential`](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Sequential.html),
[`nn.Linear`](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Linear.html),
[`nn.ReLU`](https://docs.pytorch.org/docs/2.14/generated/torch.nn.ReLU.html),
[`torch.optim.Adam`](https://docs.pytorch.org/docs/2.14/generated/torch.optim.Adam.html),
[`torch.no_grad`](https://docs.pytorch.org/docs/2.14/generated/torch.no_grad.html).
Recuerden utilizar la sintaxis `.to("cuda")` para mandar al modelo al GPU!
<!-- [``]() -->

Monten un modelo de red neuronal con 2 capas ocultas de 64 y 32 neuronas. Monten otro modelo con una sola capa oculta de 128 neuronas. Entrenen al modelo con 10 épocas. Comparen los algoritmos usando tasas de aprendizaje (parámetro `lr` en Adam) de 0.01, 0.001, 0.0001.

**Nota conceptual importante:** Los parámetros del modelo son objetos que no sólo cuentan con el valor del parámetro, si no que con otras propiedades también (a cuales otros parámetros están conectados, cuales operaciones se han realizado sobre estos parámetros, e, importantemente, cual es el valor actual del gradiente para este parámetro). Los objetos del optimizador (ADAM), el modelo y la función de pérdida tienen acceso para leer y cambiar estos parámetros.

Ejemplo de cómo se podría ver el entrenamiento de una época:


In [ ]:
model = ...
loss_fn = nn.CrossEntropyLoss()
optimizer = ...

@torch.no_grad() # evitar consumo innecesario de memoria si no se necesitan gradientes aquí
def predict(model, x):
    ...

for xb, yb in train_loader:
    logits = model(xb)
    loss = loss_fn(logits, yb)

    optimizer.zero_grad()   # settear gradiente de parámetros a 0 (para acumularlo ahí en el backward pass)
    loss.backward()         # calcula gradiente con backpropagation, escribe el gradiente en los objetos de parámetros
    optimizer.step()        # cambia los parámetros usando los valores de gradientes asociados

    # ... acumular pérdida total para reportar la pérdida total obtenida en esta época
    # ... evaluar con datos de evaluación, utilizando la función predict

